# 03 · Faces vs Places2: mask geometry, coverage and resampling strategy, with LPIPS

**Question.** How do the dataset domain (aligned faces vs. scene images), the mask geometry and coverage, and the resampling strategy affect inpainting quality? LPIPS is a primary perceptual metric here, reported alongside pixel metrics.

This notebook **extends** the project without changing it. Notebooks 01/02, `src/repaint.py`, the budget-aware sampler and the golden regression tests are untouched. All sampling goes through `src.adaptive_repaint.run_comparison`; no sampler code lives in this notebook.

| Dataset | Images | Model | Notes |
|---|---|---|---|
| Faces | `data/celeba` (CelebA, resized 256×256 by the project loader) | `google/ddpm-celebahq-256` (diffusers, ε-prediction, fixed variance) | Continuity with notebooks 01/02. Trained on CelebA-HQ; CelebA inputs are a domain shift (alignment, crop, resolution). Not the paper's checkpoint. |
| Places2 | Places2 test images (official RePaint `gts/p256`, or Places365 `val_256`) | Official RePaint `places256_300000.pt` (guided-diffusion UNet, 553 M params, learned variance) | Loaded from the official code through `src.guided_diffusion_adapter`. The learned-variance channels are **discarded**; the shared sampler uses the fixed posterior variance for both models (documented approximation). |

**Samplers (same image, mask, initial latent and noise seed):** `no_resampling` (r = 1), `fixed` RePaint, `uniform` (budget-matched, pilot charged), `adaptive` (pilot-guided). Uniform and adaptive have identical total NFE, counting the pilot. NFE and wall-clock time are reported separately.

**Masks:** 9 families × target missing ratios 10/25/40/60 % (a 10 % "large rectangle" is marked not applicable). Masks depend on (family, ratio, image index, seed) only, so image *i* of both datasets gets the same mask. The achieved coverage is measured and recorded.

**Run order.**
0. Setup and asset checks, with opt-in downloads.
1. LPIPS initialisation. A failure is reported, never silently dropped.
2. Mask families: coverage table and gallery (no model needed).
3. Model compatibility checks.
4. Smoke A: fake denoisers, no checkpoints (seconds).
5. Smoke B: real checkpoints, 1 image per dataset, tiny budget.
6. Full study: explicit opt-in, cost estimate first, resumable.
7. Analysis from saved CSV files only.

Methods, compatibility findings and limitations: `Docs/faces_places2_mask_lpips_study.md`.

## 0 · Setup

In [ ]:
# Find the repository root and make `src` importable (the notebook lives in Notebooks/).
# Override with the REPAINT_PROJECT_ROOT environment variable (e.g. on Kaggle).
import os
import sys
from pathlib import Path

def _find_project_root():
    env = os.environ.get("REPAINT_PROJECT_ROOT")
    candidates = ([Path(env).expanduser()] if env else []) + [Path.cwd(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "repaint.py").is_file() and (candidate / "configs" / "faces_places2_study.yaml").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Project root not found (needs src/repaint.py and configs/faces_places2_study.yaml). "
                            "Start Jupyter inside the repository or set REPAINT_PROJECT_ROOT.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
# Versions and hardware. Nothing is installed here (see the opt-in cell below).
import importlib
import platform
import torch

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__, "| CUDA build:", torch.version.cuda, "| CUDA usable:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")
for name in ("diffusers", "torchvision", "torchmetrics", "lpips", "scipy", "matplotlib", "pandas", "yaml"):
    try:
        module = importlib.import_module(name)
        print(f"{name:12s}:", getattr(module, "__version__", "installed"))
    except Exception as exc:
        print(f"{name:12s}: NOT AVAILABLE ({type(exc).__name__})")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", DEVICE)

In [ ]:
# ---------------------------- run switches ----------------------------------
RUN_SMOKE_FAKE = True          # §4: no checkpoints needed
RUN_SMOKE_REAL = False         # §5: needs the checkpoints, data and LPIPS weights
RUN_FULL_STUDY = False         # §6: long; set deliberately
FULL_BUDGET_PROFILE = "paper"  # "paper" (T=250, j=10, r=10), "reduced" or "smoke" (configs/faces_places2_study.yaml)
MAX_RUNS_THIS_SESSION = None   # e.g. 40 to split the full study across Kaggle sessions (resumable)
TIME_BUDGET_HOURS = 11.0       # stop starting new runs after this many hours
ALLOW_RANDOM_LPIPS_FOR_SMOKE = False  # software test only: random-backbone LPIPS, marked invalid
# ------------------------------------------------------------------------------

import json
from src.faces_places2_study import load_study_config, budget_settings, check_budget, first_existing

CFG = load_study_config(PROJECT_ROOT)
STUDY = CFG["study"]
RESULTS_ROOT = PROJECT_ROOT / STUDY["results_dir"]
print("Study config :", PROJECT_ROOT / "configs" / "faces_places2_study.yaml")
print("Results root :", RESULTS_ROOT)
print("Seeds        : masks", STUDY["mask_base_seed"], "| x_T", STUDY["latent_seed"], "+ image index | noise",
      STUDY["noise_seed"], "+ image index")
for profile in CFG["budgets"]:
    b = budget_settings(CFG, profile); plan = check_budget(b)
    print(f"Budget {profile:8s}: T={b['num_inference_steps']}, j={b['jump_length']}, r={b['jump_n_sample']} | "
          f"fixed {plan.fixed_schedule_nfe} NFE | pilot {plan.pilot_nfe} + base {plan.base_trajectory_nfe} + "
          f"{plan.extra_cycles} cycles x {plan.nfe_per_cycle} = {plan.budgeted_total_nfe} NFE "
          f"(limits {b['min_extra_cycles']}-{b['max_extra_cycles']})")

In [ ]:
# Asset check: every path is reported, nothing is downloaded here.
from src.study_datasets import resolve_data_dir

def asset_report():
    rows = []
    for name, d in CFG["datasets"].items():
        data_dir = resolve_data_dir(d["data_dirs"], PROJECT_ROOT)
        rows.append({"dataset": name, "asset": "images", "found": data_dir is not None,
                     "path": str(data_dir) if data_dir else " | ".join(map(str, d["data_dirs"]))})
        if d["backend"] == "repaint_guided_diffusion":
            ck = first_existing(d["checkpoint_candidates"], PROJECT_ROOT, want_dir=False)
            code_dir = first_existing(d["official_code_dirs"], PROJECT_ROOT, want_dir=True)
            rows.append({"dataset": name, "asset": "checkpoint", "found": ck is not None,
                         "path": str(ck) if ck else " | ".join(d["checkpoint_candidates"])})
            rows.append({"dataset": name, "asset": "official RePaint code", "found": code_dir is not None,
                         "path": str(code_dir) if code_dir else " | ".join(d["official_code_dirs"])})
        else:
            rows.append({"dataset": name, "asset": "checkpoint", "found": None,
                         "path": f"Hugging Face id {d['model_id']} (downloaded/cached by diffusers on load)"})
    return rows

import pandas as pd
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(asset_report()))

### Optional, explicit setup (all switches default to `False`)

* `CLONE_OFFICIAL_REPAINT` clones https://github.com/andreas128/RePaint into `external/RePaint`. Only its `guided_diffusion` network code is imported, to load the Places2 checkpoint.
* `INSTALL_LPIPS` runs `pip install lpips --no-deps` (pure Python; it needs scipy, which Kaggle images include). It never touches PyTorch/CUDA.
* `DOWNLOAD_REPAINT_ASSETS` runs the official `download.sh` Google-Drive IDs with `gdown` (≈2.1 GB per checkpoint, plus `data.zip`). If Drive's quota blocks it, download manually and attach the files as a Kaggle dataset. One of the two Drive IDs is the CelebA-HQ model, the other the Places2 model; files are kept under their own names and only `places256_300000.pt` is used.

In [ ]:
CLONE_OFFICIAL_REPAINT = False
INSTALL_LPIPS = False
DOWNLOAD_REPAINT_ASSETS = False

import subprocess
EXTERNAL = PROJECT_ROOT / "external"
if CLONE_OFFICIAL_REPAINT and not (EXTERNAL / "RePaint" / "guided_diffusion").exists():
    EXTERNAL.mkdir(exist_ok=True)
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/andreas128/RePaint.git",
                        str(EXTERNAL / "RePaint")], capture_output=True, text=True)
    print("git clone:", "ok" if r.returncode == 0 else r.stderr[-800:])
if INSTALL_LPIPS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "lpips", "--no-deps"], capture_output=True, text=True)
    print(r.stdout[-400:] or r.stderr[-800:])
if DOWNLOAD_REPAINT_ASSETS:
    target = EXTERNAL / "RePaint" / "data" / "pretrained"
    target.mkdir(parents=True, exist_ok=True)
    import gdown  # explicit, user-requested dependency
    for fid in ["1norNWWGYP3EZ_o05DmoW1ryKuKMmhlCX", "1QEl-btGbzQz6IwkXiFGd49uQNTUtTHsk"]:   # from official download.sh
        print(gdown.download(f"https://drive.google.com/uc?id={fid}", output=str(target) + "/", quiet=False))
    z = gdown.download("https://drive.google.com/uc?id=1Q_dxuyI41AAmSv9ti3780BwaJQqwvwMv",
                       output=str(EXTERNAL / "RePaint") + "/", quiet=False)
    if z:
        import zipfile
        with zipfile.ZipFile(z) as zf:
            zf.extractall(EXTERNAL / "RePaint" / "data")
if any([CLONE_OFFICIAL_REPAINT, INSTALL_LPIPS, DOWNLOAD_REPAINT_ASSETS]):
    display(pd.DataFrame(asset_report()))
else:
    print("No optional setup requested.")

## 1 · LPIPS (required metric)

Implementation: the `lpips` package (the official PerceptualSimilarity code of Zhang et al., CVPR 2018), v0.1 linear weights, AlexNet backbone, [-1, 1] inputs. Lower means more perceptually similar to the ground truth. It is **not** a measure of semantic correctness or human preference.

* `lpips_full`: LPIPS between the completed (composite) image and the original.
* `lpips_missing_masked`: the spatial LPIPS map (per-layer distance maps upsampled to 256×256 and summed) averaged over the missing pixels only. Nothing is cropped. Receptive fields extend beyond the hole, so this is a hole-weighted perceptual distance, not LPIPS of the hole in isolation.

If initialisation fails, the failure is printed and stored with the results. The full study refuses to run without valid LPIPS.

In [ ]:
from src.lpips_metric import load_lpips, lpips_status, LPIPSUnavailableError

LPIPS_EVAL, LPIPS_ERROR = None, None
try:
    LPIPS_EVAL = load_lpips(net=STUDY.get("lpips_net", "alex"), device=DEVICE)
except LPIPSUnavailableError as exc:
    LPIPS_ERROR = exc
    print("LPIPS COULD NOT BE INITIALISED:\n ", exc)
    if ALLOW_RANDOM_LPIPS_FOR_SMOKE:
        try:
            LPIPS_EVAL = load_lpips(net=STUDY.get("lpips_net", "alex"), device=DEVICE, random_backbone=True)
            print("Using a RANDOM-backbone LPIPS for the software smoke test only; its values are invalid.")
        except LPIPSUnavailableError as exc2:
            print("Random-backbone LPIPS also failed:", exc2)
LPIPS_META = lpips_status(LPIPS_EVAL, LPIPS_ERROR)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
(RESULTS_ROOT / "lpips_status.json").write_text(json.dumps(LPIPS_META, indent=2) + "\n")
print(json.dumps(LPIPS_META, indent=2))

## 2 · Mask families and coverage (no model needed)

Target vs achieved missing fraction for every applicable family × ratio, for the image indices used by the study. The gallery shows image index 0. The same masks are used for both datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.mask_families import coverage_table, generate_mask, mask_conditions

MASKS = CFG["masks"]
SIZE = int(STUDY["image_size"])
N_INDICES = max(d.get("max_images") or 1 for d in CFG["datasets"].values())
cov = pd.DataFrame(coverage_table(MASKS["families"], MASKS["target_ratios"], SIZE, range(N_INDICES),
                                  int(STUDY["mask_base_seed"]), MASKS.get("not_applicable")))
cov.to_csv(RESULTS_ROOT / "mask_coverage.csv", index=False)
summary = cov.groupby(["mask_family", "mask_target_ratio"])["mask_actual_ratio"].agg(["mean", "min", "max"])
display(summary.unstack("mask_target_ratio").round(3))
print(f"Max |actual - target| = {cov.mask_coverage_error.abs().max():.4f} over {len(cov)} masks; saved mask_coverage.csv")

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 8,
                     "axes.spines.top": False, "axes.spines.right": False})
FIG_ROOT = RESULTS_ROOT / "figures"; FIG_ROOT.mkdir(parents=True, exist_ok=True)
fams, ratios = MASKS["families"], MASKS["target_ratios"]
fig, axes = plt.subplots(len(ratios), len(fams), figsize=(1.35 * len(fams), 1.5 * len(ratios)))
for c, fam in enumerate(fams):
    for r, ratio in enumerate(ratios):
        ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([])
        if (fam, float(ratio)) in mask_conditions([fam], [ratio], MASKS.get("not_applicable")):
            spec = generate_mask(fam, ratio, SIZE, 0, int(STUDY["mask_base_seed"]))
            ax.imshow(spec.mask[0, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_xlabel(f"{spec.actual_ratio:.1%}", fontsize=7)
        else:
            ax.text(0.5, 0.5, "n/a", ha="center", va="center"); ax.set_frame_on(False)
        if r == 0: ax.set_title(fam.replace("_", "\n"), fontsize=7)
        if c == 0: ax.set_ylabel(f"target {ratio:.0%}")
fig.suptitle("Mask families (white = missing); x-label = achieved coverage, image index 0")
fig.savefig(FIG_ROOT / "masks_gallery.png"); plt.show()

## 3 · Model compatibility checks

Verified from the official code and configs (details in `Docs/faces_places2_mask_lpips_study.md`):

| Property | Faces: `google/ddpm-celebahq-256` | Places2: RePaint `places256_300000.pt` | Shared sampler needs |
|---|---|---|---|
| Architecture | diffusers `UNet2DModel` (≈114 M) | guided-diffusion `UNetModel`, 256 ch, attention 32/16/8, scale-shift norm, resblock up/down (552.8 M measured) | `model(x, t).sample` with 3 channels → adapter for Places2 |
| Resolution | 256 | 256 (`image_size: 256`) | 256 |
| Noise schedule | linear β 1e-4 → 0.02, 1000 steps | `noise_schedule: linear`, `diffusion_steps: 1000` (same β) | ᾱ from the scheduler |
| Prediction | ε | ε (`predict_xstart: false`) | ε |
| Variance | fixed small | **learned range** (`learn_sigma: true`, 6 output channels) | fixed posterior variance; Places2 learned variance discarded |
| Timestep input | integer 0-999 | integer 0-999 (`rescale_timesteps: false`) | `level × stride` |

The cell below re-checks these at run time on the loaded models (output shape and finiteness, ε prediction, schedule match, respacing divisibility) and times one call. Missing assets are reported, not skipped silently.

In [ ]:
from src.faces_places2_study import load_dataset_model
from src.guided_diffusion_adapter import check_model_compatibility

MODELS, COMPAT = {}, {}
LOAD_REAL_MODELS = RUN_SMOKE_REAL or RUN_FULL_STUDY
for name, dcfg in CFG["datasets"].items():
    if not LOAD_REAL_MODELS:
        COMPAT[name] = {"status": "not loaded (RUN_SMOKE_REAL and RUN_FULL_STUDY are False)"}
        continue
    try:
        model, scheduler, info = load_dataset_model(dcfg, PROJECT_ROOT, DEVICE)
        report = check_model_compatibility(model, scheduler, SIZE, budget_settings(CFG, FULL_BUDGET_PROFILE)["num_inference_steps"], DEVICE)
        MODELS[name] = (model, scheduler, info)
        COMPAT[name] = {"status": "loaded", **info, **report}
    except Exception as exc:
        COMPAT[name] = {"status": f"FAILED: {type(exc).__name__}: {exc}"}
(RESULTS_ROOT / "model_compatibility.json").write_text(json.dumps(COMPAT, indent=2, default=str) + "\n")
for name, rep in COMPAT.items():
    print(f"--- {name}")
    for k in ("status", "model_identifier", "parameters_million", "learned_variance_discarded", "prediction_type",
              "beta_schedule", "alphas_cumprod_max_abs_diff_vs_guided_diffusion_linear", "output_shapes",
              "output_finite", "seconds_per_call", "compatible"):
        if k in rep: print(f"  {k}: {rep[k]}")

## 4 · Smoke test A: fake denoisers, synthetic images (no checkpoints)

Exercises the whole study pipeline (masks, all four samplers, NFE checks, LPIPS if available, saving, aggregation, figures) on 64×64 synthetic images with a fake denoiser. The images and metrics carry **no information** about real inpainting quality.

In [ ]:
import copy
from PIL import Image
from src.faces_places2_study import run_study, write_study_summaries
from src.guided_diffusion_adapter import make_linear_ddpm_scheduler
from types import SimpleNamespace

class FakeDenoiser:
    def __call__(self, sample, timesteps):
        t = timesteps.to(sample.dtype).view(-1, 1, 1, 1) / 1000.0
        return SimpleNamespace(sample=0.5 * torch.tanh(sample) + 0.1 * t * sample.roll(1, dims=-1))
    def parameters(self): return iter(())

SMOKE_FAKE_ROOT = RESULTS_ROOT / "smoke_fake"
if RUN_SMOKE_FAKE:
    fake_cfg = copy.deepcopy(CFG); fake_cfg["study"]["image_size"] = 64
    fake_cfg["masks"]["families"] = ["center_square", "irregular_brush", "stripes_horizontal", "border_frame"]
    fake_cfg["masks"]["target_ratios"] = [0.25, 0.6]
    datasets = {}
    for d, name in enumerate(CFG["datasets"]):
        img_dir = SMOKE_FAKE_ROOT / "_synthetic_inputs" / name; img_dir.mkdir(parents=True, exist_ok=True)
        images = []
        for i in range(2):
            g = np.random.default_rng(100 * d + i)
            yy, xx = np.mgrid[0:64, 0:64] / 63.0
            base = np.stack([xx, yy, (xx + yy) / 2], -1) if d == 0 else np.stack([np.sin(8 * xx), yy, np.cos(5 * yy)], -1) * 0.5 + 0.5
            arr = (np.clip(base + 0.05 * g.standard_normal(base.shape), 0, 1) * 255).astype(np.uint8)
            path = img_dir / f"synthetic_{i}.png"; Image.fromarray(arr).save(path); images.append(path)
        datasets[name] = {"model": FakeDenoiser(), "scheduler": make_linear_ddpm_scheduler(1000),
                          "model_info": {"backend": "fake", "model_identifier": f"FakeDenoiser ({name} stand-in)",
                                         "learned_variance_discarded": False},
                          "config": CFG["datasets"][name], "images": images, "image_root": img_dir}
    manifest = run_study(datasets, fake_cfg, budget_settings(CFG, "smoke"), SMOKE_FAKE_ROOT, "cpu",
                         LPIPS_EVAL, LPIPS_META, require_lpips=False, overwrite=True, log=lambda *a: None)
    print(f"Smoke A: {manifest['completed_this_call']} runs in {manifest['elapsed_s']:.1f} s; LPIPS status: {LPIPS_META['lpips_status']}")
    print("Summaries:", list(write_study_summaries(SMOKE_FAKE_ROOT)))
else:
    print("Smoke A skipped.")

## 5 · Smoke test B: real checkpoints, one image per dataset, tiny budget

Uses the `smoke` budget (T = 50, j = 5, r = 3: 140 NFE for fixed, uniform and adaptive) on 2 mask families at 25 %. It checks the real models end to end; the budget is far too small for quality conclusions. A dataset whose assets are missing is reported and skipped explicitly.

In [ ]:
from src.study_datasets import dataset_images

def build_datasets(max_images=None, names=None):
    out = {}
    for name, dcfg in CFG["datasets"].items():
        if names and name not in names: continue
        if name not in MODELS:
            print(f"[{name}] SKIPPED: model not loaded ({COMPAT.get(name, {}).get('status')})"); continue
        try:
            cfg_d = dict(dcfg, max_images=max_images if max_images is not None else dcfg.get("max_images"))
            images = dataset_images(cfg_d, PROJECT_ROOT)
        except FileNotFoundError as exc:
            print(f"[{name}] SKIPPED: {exc}"); continue
        model, scheduler, info = MODELS[name]
        out[name] = {"model": model, "scheduler": scheduler, "model_info": info, "config": dcfg, "images": images,
                     "image_root": resolve_data_dir(dcfg["data_dirs"], PROJECT_ROOT)}
        print(f"[{name}] {len(images)} image(s) from {out[name]['image_root']}")
    return out

SMOKE_REAL_ROOT = RESULTS_ROOT / "smoke_real"
if RUN_SMOKE_REAL:
    real = build_datasets(max_images=1)
    if real:
        run_study(real, CFG, budget_settings(CFG, "smoke"), SMOKE_REAL_ROOT, DEVICE, LPIPS_EVAL, LPIPS_META,
                  families=["center_square", "irregular_brush"], ratios=[0.25], require_lpips=False, overwrite=True)
        print("Summaries:", list(write_study_summaries(SMOKE_REAL_ROOT)))
else:
    print("Smoke B skipped (RUN_SMOKE_REAL = False).")

## 6 · Full study (explicit opt-in)

A cost estimate is printed first, using the per-call time measured in §3. Per image and mask condition the four samplers execute `fixed + no_resampling + pilot + 2 × final` denoiser calls: 2,410 + 250 + 250 + 2 × 2,160 = **7,230 NFE** with the paper budget. All 35 mask conditions × 5 images × 2 datasets come to about 2.5 M calls, which is far beyond one Kaggle session. Reduce `families`, `target_ratios` or `max_images` in the config, use the `reduced` budget, and/or set `MAX_RUNS_THIS_SESSION`. Runs are resumable: finished folders are skipped, and folders from different settings are refused.

LPIPS must be valid here (`require_lpips: true`).

In [ ]:
from src.faces_places2_study import estimate_study_cost, executed_nfe_per_run

FULL_ROOT = RESULTS_ROOT / f"full_{FULL_BUDGET_PROFILE}"
if RUN_FULL_STUDY:
    budget = budget_settings(CFG, FULL_BUDGET_PROFILE)
    full = build_datasets()
    n_cond = len(mask_conditions(MASKS["families"], MASKS["target_ratios"], MASKS.get("not_applicable")))
    spc = {n: COMPAT[n].get("seconds_per_call") for n in full}
    display(pd.DataFrame(estimate_study_cost({n: len(v["images"]) for n, v in full.items()}, n_cond, budget,
                                             STUDY["methods"], spc)))
    run_study(full, CFG, budget, FULL_ROOT, DEVICE, LPIPS_EVAL, LPIPS_META, require_lpips=STUDY.get("require_lpips", True),
              max_runs=MAX_RUNS_THIS_SESSION, time_budget_hours=TIME_BUDGET_HOURS)
    print("Summaries:", list(write_study_summaries(FULL_ROOT)))
else:
    print("Full study not run (RUN_FULL_STUDY = False).")

## 7 · Analysis (from saved CSV files only)

Uses the most complete results available: full study, then smoke B, then smoke A. Figures go to `<results>/figures/`. Raw metric values are compared **within** a dataset only. Across datasets the notebook compares within-dataset quantities: mask-difficulty ranks, change relative to `no_resampling`, and paired adaptive − uniform differences.

In [ ]:
from src.faces_places2_study import collect_results, write_study_summaries

choices = [("full", FULL_ROOT), ("smoke_real", SMOKE_REAL_ROOT), ("smoke_fake", SMOKE_FAKE_ROOT)]
KIND, ROOT = next(((k, r) for k, r in choices if r.exists() and any(r.rglob("study_rows.csv"))), (None, None))
if ROOT is None:
    raise FileNotFoundError("No results yet; run §4, §5 or §6.")
write_study_summaries(ROOT)
DF = pd.read_csv(ROOT / "per_image_results.csv")
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
LPIPS_OK = DF["lpips_full"].notna().any() and bool(DF["lpips_valid"].fillna(False).astype(bool).all())
PRIMARY = "lpips_full" if DF["lpips_full"].notna().any() else "missing_mae"
print(f"Results: {KIND} ({ROOT}) | {len(DF)} rows | datasets {sorted(DF.dataset.unique())} | "
      f"LPIPS status {sorted(DF.lpips_status.astype(str).unique())}")
if KIND == "smoke_fake":
    print("NOTE: fake-denoiser results. The figures only demonstrate the pipeline; they say nothing about inpainting quality.")
if not LPIPS_OK:
    print("NOTE: LPIPS values are missing or come from a random test backbone; LPIPS figures are NOT valid measurements.")

METHODS = [m for m in ["no_resampling", "fixed", "uniform", "adaptive"] if m in set(DF.method)]
LABEL = {"no_resampling": "No resampling", "fixed": "Fixed RePaint", "uniform": "Uniform", "adaptive": "Adaptive"}
COLOR = {"fixed": "#2a78d6", "no_resampling": "#eb6834", "uniform": "#1baf7a", "adaptive": "#eda100"}
MARK = {"no_resampling": "s", "fixed": "o", "uniform": "^", "adaptive": "D"}
DATASETS = sorted(DF.dataset.unique())
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True, "font.size": 8})

### 7.1 Galleries: original / mask / masked input / completed images (composites)

In [ ]:
def gallery(dataset, ratio=None, image_id=None, families=None):
    sub = DF[DF.dataset == dataset]
    ratio = ratio if ratio is not None else sorted(sub.mask_target_ratio.unique())[len(sub.mask_target_ratio.unique()) // 2]
    image_id = image_id or sorted(sub.image_id.unique())[0]
    sub = sub[(sub.mask_target_ratio == ratio) & (sub.image_id == image_id)]
    fams = [f for f in (families or MASKS["families"]) if f in set(sub.mask_family)]
    if not fams:
        print("nothing to show"); return
    cols = ["original", "mask", "masked input"] + [LABEL[m] for m in METHODS]
    fig, axes = plt.subplots(len(fams), len(cols), figsize=(1.45 * len(cols), 1.5 * len(fams) + 0.4), squeeze=False)
    for r, fam in enumerate(fams):
        run_dir = Path(sub[sub.mask_family == fam].run_dir.iloc[0])
        files = ["original.png", "mask_missing_white.png", "masked_input.png"] + [f"{m}_composite.png" for m in METHODS]
        actual = sub[sub.mask_family == fam].mask_actual_ratio.iloc[0]
        for c, f in enumerate(files):
            ax = axes[r, c]; ax.imshow(Image.open(run_dir / "images" / f), cmap="gray" if "mask" in f else None)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            if r == 0: ax.set_title(cols[c], fontsize=7)
        axes[r, 0].set_ylabel(f"{fam}\n{actual:.1%}", fontsize=7)
    fig.suptitle(f"{dataset} · {image_id} · target {ratio:.0%} · completed = sampler output with known pixels restored")
    fig.savefig(FIG / f"gallery_{dataset}_{ratio:.2f}.png"); plt.show()

for d in DATASETS:
    gallery(d)

### 7.2 Metric tables, kept separate per dataset

In [ ]:
TABLE_METRICS = [m for m in ["lpips_full", "lpips_missing_masked", "composite_full_psnr_db", "composite_full_ssim",
                               "composite_full_mse", "missing_mae", "missing_mse", "missing_psnr_db",
                               "boundary_inner_psnr_db", "raw_known_mae", "total_nfe", "total_wall_time_s"]
                 if m in DF and DF[m].notna().any()]
for d in DATASETS:
    t = DF[DF.dataset == d].groupby(["mask_family", "mask_target_ratio", "method"])[TABLE_METRICS].mean()
    t.insert(0, "n_images", DF[DF.dataset == d].groupby(["mask_family", "mask_target_ratio", "method"]).size())
    t.to_csv(ROOT / f"table_{d}.csv")
    print(f"=== {d} (means over images; saved table_{d}.csv)")
    display(t.round(4))

### 7.3 LPIPS (or the primary metric) vs achieved mask coverage

In [ ]:
for d in DATASETS:
    sub = DF[DF.dataset == d]
    fams = [f for f in MASKS["families"] if f in set(sub.mask_family)]
    ncol = min(5, len(fams)); nrow = int(np.ceil(len(fams) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.3 * nrow), squeeze=False, sharey=True)
    for i, fam in enumerate(fams):
        ax = axes[i // ncol, i % ncol]
        g = sub[sub.mask_family == fam].groupby(["method", "mask_target_ratio"]).agg(
            x=("mask_actual_ratio", "mean"), y=(PRIMARY, "mean"), s=(PRIMARY, "std")).reset_index()
        for m in METHODS:
            gm = g[g.method == m].sort_values("x")
            ax.errorbar(gm.x * 100, gm.y, yerr=gm.s.fillna(0), color=COLOR[m], marker=MARK[m], ms=4, lw=1.5,
                        capsize=2, label=LABEL[m])
        ax.set_title(fam, fontsize=8); ax.set_xlabel("achieved coverage (%)")
        if i % ncol == 0: ax.set_ylabel(f"{PRIMARY} ({'lower' if 'lpips' in PRIMARY or 'mae' in PRIMARY else 'higher'} is better)")
    for j in range(len(fams), nrow * ncol): axes[j // ncol, j % ncol].set_visible(False)
    axes[0, 0].legend(fontsize=7, frameon=False)
    fig.suptitle(f"{d}: {PRIMARY} vs coverage (mean ± sd over images)" + ("" if LPIPS_OK else " — LPIPS NOT VALID"))
    plt.tight_layout(); fig.savefig(FIG / f"coverage_{d}_{PRIMARY}.png"); plt.show()

### 7.4 Mask type × sampler (means over images and coverage levels)

In [ ]:
for metric in [m for m in (PRIMARY, "missing_psnr_db") if m in DF]:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.2 * len(DATASETS), 0.38 * len(MASKS["families"]) + 1.6), squeeze=False)
    for ax, d in zip(axes[0], DATASETS):
        p = DF[DF.dataset == d].pivot_table(index="mask_family", columns="method", values=metric, aggfunc="mean")
        p = p[[m for m in METHODS if m in p]]
        im = ax.imshow(p.values, cmap="Blues" if metric in ("lpips_full", "missing_mae", "lpips_missing_masked") else "Blues_r", aspect="auto")
        ax.set_xticks(range(p.shape[1]), [LABEL[m] for m in p.columns], rotation=20); ax.set_yticks(range(p.shape[0]), p.index)
        for (r, c), v in np.ndenumerate(p.values):
            ax.text(c, r, f"{v:.3f}" if abs(v) < 10 else f"{v:.1f}", ha="center", va="center", fontsize=6,
                    color="white" if im.norm(v) > 0.6 else "#222")
        ax.set_title(f"{d}: {metric}"); ax.grid(False)
        fig.colorbar(im, ax=ax, shrink=0.8)
    fig.suptitle("Darker = worse. Colour scales are per dataset; compare within a panel only."
                 + ("" if LPIPS_OK or "lpips" not in metric else "  LPIPS NOT VALID"))
    plt.tight_layout(); fig.savefig(FIG / f"heatmap_{metric}.png"); plt.show()

### 7.5 Paired adaptive − uniform (same image, mask, seeds and NFE)

In [ ]:
from src.faces_places2_study import paired_differences
pair_metrics = [m for m in (PRIMARY, "missing_psnr_db") if m in DF]
PAIRED = paired_differences(DF, "adaptive", "uniform", pair_metrics)
PAIRED.to_csv(ROOT / "paired_adaptive_vs_uniform.csv", index=False)
fig, axes = plt.subplots(len(pair_metrics), len(DATASETS), figsize=(5 * len(DATASETS), 3.2 * len(pair_metrics)), squeeze=False)
for r, metric in enumerate(pair_metrics):
    for c, d in enumerate(DATASETS):
        ax = axes[r, c]
        p = PAIRED[(PAIRED.dataset == d) & (PAIRED.metric == metric)].reset_index(drop=True)
        labels = [f"{f} {q:.0%}" for f, q in zip(p.mask_family, p.mask_target_ratio)]
        ax.errorbar(p.mean_diff, range(len(p)), xerr=p.sem_diff.fillna(0), fmt="o", color=COLOR["adaptive"], ms=4, capsize=2)
        ax.axvline(0, color="#555", lw=1)
        ax.set_yticks(range(len(p)), labels, fontsize=6); ax.invert_yaxis()
        better = "lower" if metric in ("lpips_full", "lpips_missing_masked", "missing_mae") else "higher"
        ax.set_xlabel(f"adaptive − uniform {metric} (mean ± SEM; {better} = adaptive better)")
        ax.set_title(f"{d}: {int(p.n_pairs.sum()) if len(p) else 0} pairs"
                     + ("" if LPIPS_OK or "lpips" not in metric else " (LPIPS NOT VALID)"))
plt.tight_layout(); fig.savefig(FIG / "paired_adaptive_vs_uniform.png"); plt.show()
display(PAIRED.round(4))

### 7.6 Cross-dataset comparison (within-dataset quantities only)

In [ ]:
from src.faces_places2_study import mask_difficulty_ranking, relative_to_no_resampling
RANK = mask_difficulty_ranking(DF, "fixed", PRIMARY) if "fixed" in METHODS else pd.DataFrame()
if len(RANK):
    wide = RANK.pivot(index="mask_family", columns="dataset", values="rank_most_challenging")
    display(wide.sort_values(DATASETS[0]))
    if len(DATASETS) == 2:
        fig, ax = plt.subplots(figsize=(4.2, 0.32 * len(wide) + 1.5))
        for fam, row in wide.iterrows():
            ax.plot([0, 1], [row[DATASETS[0]], row[DATASETS[1]]], color="#2a78d6", marker="o", lw=1.2)
            ax.text(-0.04, row[DATASETS[0]], fam, ha="right", va="center", fontsize=7)
            ax.text(1.04, row[DATASETS[1]], fam, ha="left", va="center", fontsize=7)
        ax.set_xticks([0, 1], DATASETS); ax.set_yticks(range(1, len(wide) + 1)); ax.invert_yaxis()
        ax.set_xlim(-0.8, 1.8); ax.set_ylabel("rank (1 = most challenging)")
        ax.set_title(f"Mask difficulty rank by {PRIMARY}, fixed RePaint" + ("" if LPIPS_OK else "\n(LPIPS NOT VALID)")); ax.grid(False)
        fig.savefig(FIG / "mask_difficulty_rank.png"); plt.show()
REL = relative_to_no_resampling(DF, PRIMARY) if "no_resampling" in METHODS else pd.DataFrame()
if len(REL):
    agg = REL.groupby(["dataset", "method"])["mean"].mean().unstack("method")
    agg = agg[[m for m in ["fixed", "uniform", "adaptive"] if m in agg]]
    display((agg * 100).round(2).rename_axis(f"% change in {PRIMARY} vs no resampling"))
    fig, ax = plt.subplots(figsize=(5, 3)); w = 0.8 / agg.shape[1]
    for i, m in enumerate(agg.columns):
        ax.bar(np.arange(len(agg)) + (i - (agg.shape[1] - 1) / 2) * w, agg[m] * 100, w * 0.92, color=COLOR[m], label=LABEL[m])
    ax.axhline(0, color="#555", lw=1); ax.set_xticks(range(len(agg)), agg.index)
    ax.set_ylabel(f"% change in {PRIMARY} vs no resampling"); ax.legend(frameon=False, fontsize=7); ax.grid(axis="x", visible=False)
    ax.set_title("Relative effect of resampling, per dataset (mean over conditions)" + ("" if LPIPS_OK else "\nLPIPS NOT VALID"))
    fig.savefig(FIG / "relative_to_no_resampling.png"); plt.show()

### 7.7 Compute: NFE and wall-clock time (separate panels)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
agg = DF.groupby(["dataset", "method"])[["total_nfe", "total_wall_time_s"]].mean().reset_index()
for ax, col, title in zip(axes, ["total_nfe", "total_wall_time_s"], ["Denoiser evaluations per image (NFE)", "Wall-clock seconds per image"]):
    w = 0.8 / len(METHODS)
    for i, m in enumerate(METHODS):
        v = agg[agg.method == m].set_index("dataset").reindex(DATASETS)[col]
        ax.bar(np.arange(len(DATASETS)) + (i - (len(METHODS) - 1) / 2) * w, v, w * 0.92, color=COLOR[m], label=LABEL[m])
    ax.set_xticks(range(len(DATASETS)), DATASETS); ax.set_title(title); ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False, fontsize=7)
plt.tight_layout(); fig.savefig(FIG / "compute.png"); plt.show()
check = DF.pivot_table(index=["dataset", "mask_family", "mask_target_ratio", "image_id"], columns="method", values="total_nfe")
if {"uniform", "adaptive"} <= set(check.columns):
    assert (check["uniform"] == check["adaptive"]).all(), "uniform/adaptive NFE mismatch"
    print("Verified: uniform and adaptive total NFE are identical for every paired run.")

### 7.8 What the executed results support

The cell below states only what the saved numbers show and prints the evidence it is based on. With few images it reports descriptive counts, not significance. For smoke runs it refuses to draw conclusions.

In [ ]:
if KIND == "smoke_fake":
    print("Smoke A (fake denoiser): no conclusions about datasets, masks or samplers can be drawn.")
else:
    n_img = DF.groupby("dataset").image_id.nunique().to_dict()
    print(f"Run kind: {KIND}; images per dataset: {n_img}; primary metric: {PRIMARY} (LPIPS valid: {LPIPS_OK})")
    if len(RANK):
        for d in DATASETS:
            top = RANK[RANK.dataset == d].nsmallest(3, "rank_most_challenging")
            print(f"[{d}] most challenging mask families for fixed RePaint by mean {PRIMARY}: "
                  + ", ".join(f"{r.mask_family} ({r['mean']:.4f}, n={int(r['count'])})" for _, r in top.iterrows()))
    p = PAIRED[PAIRED.metric == PRIMARY]
    if len(p):
        better_col = [c for c in p.columns if c.startswith("n_adaptive_better")][0]
        worse_col = [c for c in p.columns if c.startswith("n_uniform_better")][0]
        for d in DATASETS:
            q = p[p.dataset == d]
            lower = q.lower_is_better.iloc[0]
            favour = (q.mean_diff < 0).sum() if lower else (q.mean_diff > 0).sum()
            print(f"[{d}] adaptive vs uniform on {PRIMARY}: conditions favouring adaptive {favour}/{len(q)}; "
                  f"image pairs adaptive better {int(q[better_col].sum())}, uniform better {int(q[worse_col].sum())}, "
                  f"ties {int(q.n_ties.sum())}")
        per_ds = {d: (p[p.dataset == d].mean_diff < 0).mean() if p.lower_is_better.iloc[0] else (p[p.dataset == d].mean_diff > 0).mean()
                  for d in DATASETS}
        consistent = all(v > 0.5 for v in per_ds.values()) or all(v < 0.5 for v in per_ds.values())
        print("Direction of the adaptive − uniform effect is", "the same" if consistent else "NOT the same",
              "in both datasets (share of conditions favouring adaptive:", {k: round(v, 2) for k, v in per_ds.items()}, ").")
    print("These are descriptive statements about the executed runs only. Report spreads and image counts; "
          "small samples cannot establish a consistent improvement.")